# BERT Fine-Tuning for Word Ladder — Distance Regression

Fine-tune `bert-base-uncased` to predict the **BFS shortest-path distance** between two words in the word-ladder graph.

**Input format:** `word_a` + `word_b` (BERT sentence-pair) → predicted distance (float)

**Task:** Regression (MSE loss, 1 output neuron)

**Data:** `data/training/wordladder_english5_*.csv` from notebook 05

**Usage at inference:** For each neighbor of the current word, predict `distance(neighbor, target)`. Pick the neighbor with the lowest predicted distance → A\* heuristic.

**Run 7+ training:** 6 epochs, **cosine LR schedule**, 6% warmup (good for 600k+ rows from notebook 05). If Colab OOM, set `BATCH_SIZE = 16` or `gradient_accumulation_steps=2`.

**Dependencies:** `pip install transformers torch pandas tqdm accelerate networkx`

## 1. Setup and imports

In [2]:
# Ensure accelerate is installed (required by Trainer). Run this cell FIRST if you get ImportError.
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "accelerate>=1.1.0", "-q"])
import accelerate
print(f"accelerate {accelerate.__version__} OK")

c:\Users\doric\Documents\GitHub\word-ladder\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


accelerate 1.13.0 OK


In [3]:
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

BASE = Path("../data")
TRAINING = BASE / "training"
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 32  # two 5-letter words + special tokens is very short
BATCH_SIZE = 32
EPOCHS = 6  # Run 7+: one extra epoch; pair with cosine schedule below
SEED = 42

torch.manual_seed(SEED)
print(f"PyTorch: {torch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

PyTorch: 2.10.0+cpu
Device: cpu


## 2. Load the training data

Read the CSVs. Each row has `text_a` (word A), `text_b` (word B), and `label` (BFS shortest-path distance, integer).

In [3]:
train_df = pd.read_csv(TRAINING / "wordladder_english5_train.csv")
val_df = pd.read_csv(TRAINING / "wordladder_english5_val.csv")
test_df = pd.read_csv(TRAINING / "wordladder_english5_test.csv")

print(f"Train: {len(train_df)} examples")
print(f"Val:   {len(val_df)} examples")
print(f"Test:  {len(test_df)} examples")
print(f"Label stats (train): mean={train_df['label'].mean():.2f}, "
      f"min={train_df['label'].min()}, max={train_df['label'].max()}")
print(f"\nSample rows:")
print(train_df.head(3).to_string(index=False))

Train: 13444 examples
Val: 785 examples
Test: 771 examples
Class balance (train): 25.18% positive

Sample row:
{'text_a': 'cruse [SEP] gnash', 'text_b': 'crude', 'label': 0}


## 3. Load tokenizer and tokenize

Encode each `(word_a, word_b)` pair as a BERT sentence pair: `[CLS] word_a [SEP] word_b [SEP]`

BERT's `token_type_ids` distinguish the two words. The tokenizer handles the special tokens automatically.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_pair(text_a: str, text_b: str):
    """Tokenize (word_a, word_b) as a BERT sentence pair for distance regression."""
    return tokenizer(
        text_a,
        text_b,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors=None,  # return lists, not tensors
    )


def tokenize_df(df: pd.DataFrame):
    """Tokenize all rows in a dataframe."""
    encodings = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Tokenizing"):
        enc = tokenize_pair(str(row["text_a"]), str(row["text_b"]))
        encodings.append(enc)
    return encodings


train_enc = tokenize_df(train_df)
val_enc = tokenize_df(val_df)
test_enc = tokenize_df(test_df)
print(f"Tokenized {len(train_enc)} train, {len(val_enc)} val, {len(test_enc)} test")

Tokenizing: 100%|██████████| 771/771 [00:00<00:00, 2996.93it/s]

Tokenized 13444 train, 785 val, 771 test


## 4. Create PyTorch Dataset

Wrap tokenized data + labels into a Dataset that returns dicts of tensors.

In [5]:
class WordLadderDataset(Dataset):
    """Dataset of (input_ids, attention_mask, labels) for BERT regression."""

    def __init__(self, encodings: list, labels: list):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v) for k, v in self.encodings[idx].items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item


train_dataset = WordLadderDataset(train_enc, train_df["label"].tolist())
val_dataset = WordLadderDataset(val_enc, val_df["label"].tolist())
test_dataset = WordLadderDataset(test_enc, test_df["label"].tolist())

print(f"Train dataset: {len(train_dataset)} examples")
print(f"Sample batch keys: {list(train_dataset[0].keys())}")
print(f"Sample label (should be float): {train_dataset[0]['labels']}")

Train dataset: 13444 examples
Sample batch keys: ['input_ids', 'token_type_ids', 'attention_mask', 'labels']


## 5. Load model and set up Trainer

We use `AutoModelForSequenceClassification` with `num_labels=1` and `problem_type="regression"`. This gives a single-output regression head on top of BERT (MSE loss). The Trainer handles training, logging, and evaluation.

In [31]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=1, problem_type="regression"
)

training_args = TrainingArguments(
    output_dir="../outputs/bert_wordladder",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_ratio=0.06,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="mae",
    greater_is_better=False,
    seed=SEED,
)


def compute_metrics(eval_pred):
    import numpy as np
    preds, labels = eval_pred
    preds = preds.squeeze()
    mae = np.mean(np.abs(preds - labels))
    rmse = np.sqrt(np.mean((preds - labels) ** 2))
    return {"mae": mae, "rmse": rmse}


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print(f"Model: {MODEL_NAME} (regression, 1 output)")
print(f"Epochs: {EPOCHS}, Batch size: {BATCH_SIZE}, LR: {training_args.learning_rate}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1643.93it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider tr

Model: bert-base-uncased
Epochs: 3, Batch size: 32


## 6. Train

In [8]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.507553,0.529531,0.774522
2,0.494332,0.525537,0.755414
3,0.390879,0.561833,0.757962


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.69it/s]
c:\Users\doric\Documents\GitHub\word-ladder\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.98it/s]
c:\Users\doric\Documents\GitHub\word-ladder\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.28it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert

TrainOutput(global_step=1263, training_loss=0.45724175076397844, metrics={'train_runtime': 7748.178, 'train_samples_per_second': 5.205, 'train_steps_per_second': 0.163, 'total_flos': 1350680602690560.0, 'train_loss': 0.45724175076397844, 'epoch': 3.0})

## 7. Evaluate on validation and test

In [10]:
from torch.utils.data import DataLoader
import numpy as np

def eval_dataset(model, dataset, batch_size=32):
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            labels = batch.pop("labels")
            out = model(**{k: v.to(model.device) for k, v in batch.items()})
            preds = out.logits.squeeze(-1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.tolist())
    preds = np.array(all_preds)
    labels = np.array(all_labels)
    mae = np.mean(np.abs(preds - labels))
    rmse = np.sqrt(np.mean((preds - labels) ** 2))
    within_1 = np.mean(np.abs(preds - labels) < 1.0)
    return {"mae": mae, "rmse": rmse, "within_1": within_1}

val_result = eval_dataset(trainer.model, val_dataset)
test_result = eval_dataset(trainer.model, test_dataset)

print("\n=== Validation ===")
print(f"  MAE:  {val_result['mae']:.4f}")
print(f"  RMSE: {val_result['rmse']:.4f}")
print(f"  Within ±1 step: {val_result['within_1']:.2%}")

print("\n=== Test ===")
print(f"  MAE:  {test_result['mae']:.4f}")
print(f"  RMSE: {test_result['rmse']:.4f}")
print(f"  Within ±1 step: {test_result['within_1']:.2%}")


=== Validation ===
  accuracy: 0.7745

=== Test ===
  accuracy: 0.7951


## 8. Save the model

Save to disk so we can load it later for inference.

In [11]:
SAVE_PATH = Path("../models/bert_wordladder_5letter")
OUTPUTS = Path("../outputs/bert_wordladder")
SAVE_PATH.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(SAVE_PATH))
tokenizer.save_pretrained(str(SAVE_PATH))

# Fallback: if model.safetensors missing (e.g. Drive sync), copy from latest checkpoint
if not (SAVE_PATH / "model.safetensors").exists() and OUTPUTS.exists():
    import shutil
    ckpts = sorted(OUTPUTS.glob("checkpoint-*"))
    if ckpts:
        latest = ckpts[-1]
        for f in ["model.safetensors", "pytorch_model.bin"]:
            src = latest / f
            if src.exists():
                shutil.copy(src, SAVE_PATH)
                print(f"Copied {f} from {latest}")

print(f"Model and tokenizer saved to {SAVE_PATH}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.87it/s]

Model and tokenizer saved to ..\models\bert_wordladder_5letter


## 9. Inference

Score neighbors by predicted distance to target. Pick the neighbor with the **lowest** predicted distance at each step (A\* heuristic).

### Score candidates and generate paths

`score_candidates` predicts distance from each candidate to the target. `generate_path` uses beam search with this heuristic, falling back to BFS if stuck.

**If you have a downloaded model** (from Colab) but didn't run training above, run this first:

In [4]:
# Load saved model (skip if you just ran training)
SAVE_PATH = Path("../models/bert_wordladder_5letter")
has_weights = (SAVE_PATH / "model.safetensors").exists() or (SAVE_PATH / "pytorch_model.bin").exists()
if SAVE_PATH.exists() and has_weights:
    model = AutoModelForSequenceClassification.from_pretrained(str(SAVE_PATH))
    tokenizer = AutoTokenizer.from_pretrained(str(SAVE_PATH))
    vocab = {w.strip().lower() for w in (BASE / "islands" / "english_5_largest_island.txt").read_text(encoding="utf-8").splitlines() if w.strip()}
    print(f"Loaded model from {SAVE_PATH}, vocab size {len(vocab)}")
elif SAVE_PATH.exists():
    print(f"Folder exists but model.safetensors missing. Download from Colab and extract into {SAVE_PATH}")
else:
    print("Model not found. Run training first, or download from Colab (see docs/colab-setup.md).")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3447.39it/s]


Loaded model from ..\models\bert_wordladder_5letter, vocab size 9902


In [6]:
import string

def one_letter_neighbors(w: str, vocab: set) -> set:
    return {w[:i]+c+w[i+1:] for i in range(len(w)) for c in string.ascii_lowercase if c!=w[i]} & vocab


def score_candidates(model, tokenizer, current: str, target: str, candidates: list, device=None):
    """
    Predict distance from each candidate to target (batched for speed).
    Returns list of (candidate, predicted_distance) sorted ascending (lower = closer = better).
    """
    if device is None:
        device = next(model.parameters()).device
    if not candidates:
        return []

    enc = tokenizer(
        [cand for cand in candidates],
        [target] * len(candidates),
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="pt",
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    model.eval()
    with torch.no_grad():
        out = model(**enc)
        preds = out.logits.squeeze(-1).cpu().tolist()

    if isinstance(preds, float):
        preds = [preds]

    scores = list(zip(candidates, preds))
    return sorted(scores, key=lambda x: x[1])


# Load vocab if not already loaded (e.g. when running after training without cell 21)
if "vocab" not in dir():
    vocab = {w.strip().lower() for w in (BASE / "islands" / "english_5_largest_island.txt").read_text(encoding="utf-8").splitlines() if w.strip()}
    print(f"Loaded vocab: {len(vocab)} words")

# Quick test: rank neighbors of a word by predicted distance to a target
test_current, test_target = "crane", "flame"
test_neighbors = list(one_letter_neighbors(test_current, vocab))
ranked = score_candidates(model, tokenizer, test_current, test_target, test_neighbors)
print(f"Current: {test_current}, Target: {test_target}")
print(f"Neighbors ranked by predicted distance to target:")
for cand, dist in ranked[:10]:
    print(f"  {cand}: {dist:.2f}")

Current: crane, Target: flame
Neighbors ranked by predicted distance to target:
  crare: 2.13
  crape: 2.30
  crave: 2.70
  craze: 2.86
  crake: 2.87
  crone: 3.11
  crine: 3.28
  crate: 3.46
  crank: 5.06


In [8]:
def shortest_path_bfs(start, target, vocab):
    from collections import deque
    if start not in vocab or target not in vocab:
        return None
    queue = deque([(start, [start])])
    seen = {start}
    while queue:
        w, path = queue.popleft()
        if w == target:
            return path
        for n in one_letter_neighbors(w, vocab) - seen:
            seen.add(n)
            queue.append((n, path + [n]))
    return None


def generate_path_astar(model, tokenizer, vocab, start, target, max_expansions=300, max_steps=20):
    """
    A* search with BERT predicted distance as heuristic.
    Uses a priority queue so short paths with good heuristic values
    are expanded before longer detours — unlike beam search where all
    paths advance in lockstep.
    """
    import heapq
    start, target = start.lower(), target.lower()
    if start not in vocab or target not in vocab:
        return None, None
    device = next(model.parameters()).device

    h0 = score_candidates(model, tokenizer, start, target, [start], device)[0][1]
    h0 = max(h0, 0)

    counter = 0
    pq = [(h0, counter, [start])]
    best_g = {start: 0}
    expansions = 0

    while pq and expansions < max_expansions:
        f, _, path = heapq.heappop(pq)
        current = path[-1]
        g_current = len(path) - 1

        if current == target:
            return path, "astar"

        if g_current >= max_steps:
            continue

        if g_current > best_g.get(current, float('inf')):
            continue

        neighbors = list(one_letter_neighbors(current, vocab))
        if not neighbors:
            continue

        ranked = score_candidates(model, tokenizer, current, target, neighbors, device)
        expansions += 1

        for word, pred_dist in ranked:
            g_new = g_current + 1
            if g_new < best_g.get(word, float('inf')):
                best_g[word] = g_new
                h = max(pred_dist, 0)
                f = g_new + h
                counter += 1
                heapq.heappush(pq, (f, counter, path + [word]))

    bfs_path = shortest_path_bfs(start, target, vocab)
    return (bfs_path, "bfs") if bfs_path else (None, None)


def generate_path(model, tokenizer, vocab, start: str, target: str, max_steps=20, **kwargs):
    return generate_path_astar(model, tokenizer, vocab, start, target, max_steps=max_steps)


# Try some paths
test_pairs = [("miles", "tanks"), ("light", "right"), ("crane", "flame"), ("black", "white")]
for s, t in test_pairs:
    path, method = generate_path(model, tokenizer, vocab, s, t)
    if path:
        print(f"{s} → {t}: {' → '.join(path)} ({len(path)-1} steps, {method})")
    else:
        print(f"{s} → {t}: no path found")

miles → tanks: miles → males → manes → manks → tanks (4 steps, astar)
light → right: light → right (1 steps, astar)
crane → flame: crane → crave → clave → clame → flame (4 steps, astar)
black → white: black → brack → brick → trick → trice → trite → write → white (7 steps, bfs)


In [9]:
# === Batch evaluation: A* vs BFS on random pairs ===

import time
import random  # ✅ FIX: add this
import networkx as nx
from collections import defaultdict

PLAYABLE_PATH = BASE / "islands" / "english_5_strict_largest_island.txt"
playable = sorted({
    w.strip().lower()
    for w in PLAYABLE_PATH.read_text(encoding="utf-8").splitlines()
    if w.strip()
})

G_eval = nx.Graph()
G_eval.add_nodes_from(vocab)

for w in tqdm(vocab, desc="Building eval graph"):
    for nb in one_letter_neighbors(w, vocab):
        if w < nb:
            G_eval.add_edge(w, nb)

N_PAIRS = 200
random.seed(42)

eval_pairs = []
seen = set()

while len(eval_pairs) < N_PAIRS:
    a, b = random.sample(playable, 2)
    if (a, b) not in seen:
        seen.add((a, b))
        bfs_dist = nx.shortest_path_length(G_eval, a, b)
        if 3 <= bfs_dist <= 10:
            eval_pairs.append((a, b, bfs_dist))

print(f"Sampled {len(eval_pairs)} pairs (distance 3–10)")

results = []
t0 = time.time()

for start, target, bfs_dist in tqdm(eval_pairs, desc="A* eval"):
    path, method = generate_path(model, tokenizer, vocab, start, target)
    astar_len = len(path) - 1 if path else None

    results.append({
        "start": start,
        "target": target,
        "bfs_optimal": bfs_dist,
        "astar_steps": astar_len,
        "method": method,
        "optimal": astar_len == bfs_dist if astar_len is not None else False,
    })

elapsed = time.time() - t0

astar_success = sum(1 for r in results if r["method"] == "astar")
bfs_fallback = sum(1 for r in results if r["method"] == "bfs")
optimal_count = sum(1 for r in results if r["optimal"])

astar_results = [r for r in results if r["method"] == "astar"]

avg_astar_len = (
    sum(r["astar_steps"] for r in astar_results) / len(astar_results)
    if astar_results else 0
)

avg_bfs_opt = (
    sum(r["bfs_optimal"] for r in astar_results) / len(astar_results)
    if astar_results else 0
)

avg_ratio = avg_astar_len / avg_bfs_opt if avg_bfs_opt else 0

print(f"\n{'='*50}")
print(f"EVALUATION: {len(eval_pairs)} pairs (distance 3–10)")
print(f"{'='*50}")
print(f"A* success:     {astar_success}/{len(eval_pairs)} ({astar_success/len(eval_pairs):.1%})")
print(f"BFS fallback:   {bfs_fallback}/{len(eval_pairs)} ({bfs_fallback/len(eval_pairs):.1%})")
print(f"Optimal paths:  {optimal_count}/{len(eval_pairs)} ({optimal_count/len(eval_pairs):.1%})")
print(f"Avg A* length:  {avg_astar_len:.2f} (vs BFS optimal {avg_bfs_opt:.2f}, ratio {avg_ratio:.2f})")
print(f"Total time:     {elapsed:.1f}s ({elapsed/len(eval_pairs):.2f}s per pair)")
print()

# Breakdown by BFS distance
by_dist = defaultdict(list)
for r in results:
    by_dist[r["bfs_optimal"]].append(r)

print("Breakdown by BFS optimal distance:")
print(f"{'Dist':>4} | {'Count':>5} | {'A* OK':>5} | {'Optimal':>7} | {'Avg A* len':>10}")
print("-" * 50)

for d in sorted(by_dist):
    group = by_dist[d]
    n = len(group)
    ok = sum(1 for r in group if r["method"] == "astar")
    opt = sum(1 for r in group if r["optimal"])
    avg_len = (
        sum(r["astar_steps"] for r in group if r["method"] == "astar") / ok
        if ok else 0
    )

    print(f"{d:>4} | {n:>5} | {ok:>5} | {opt:>7} | {avg_len:>10.2f}")

Building eval graph:   0%|          | 0/9902 [00:00<?, ?it/s]

Building eval graph: 100%|██████████| 9902/9902 [00:00<00:00, 11333.93it/s]


Sampled 200 pairs (distance 3–10)


A* eval: 100%|██████████| 200/200 [5:17:35<00:00, 95.28s/it]    


EVALUATION: 200 pairs (distance 3–10)
A* success:     140/200 (70.0%)
BFS fallback:   60/200 (30.0%)
Optimal paths:  178/200 (89.0%)
Avg A* length:  6.76 (vs BFS optimal 6.59, ratio 1.03)
Total time:     19055.2s (95.28s per pair)

Breakdown by BFS optimal distance:
Dist | Count | A* OK | Optimal | Avg A* len
--------------------------------------------------
   3 |     4 |     4 |       4 |       3.00
   4 |    13 |    13 |      12 |       4.08
   5 |    24 |    23 |      22 |       5.09
   6 |    41 |    33 |      34 |       6.24
   7 |    37 |    23 |      33 |       7.22
   8 |    31 |    18 |      28 |       8.17
   9 |    33 |    19 |      30 |       9.16
  10 |    17 |     7 |      15 |      10.29
